# 🧪 Catalogue Extractor — Validation on a bigger OPEN Qwen (Groq)

**Separate from the T4 bot notebook**, so the bot work stays clean. This notebook answers one question: *can a larger open-source model rebuild the offer catalogue cleanly enough to maintain it automatically?*

- **No GPU, no re-scrape.** It reuses the cached crawl (`data/djezzy_pages.json`, committed) and calls a **hosted** open Qwen on Groq — so a plain **CPU runtime** is enough and there is **no heavy install**.
- **Your gold is never overwritten.** The extractor backs up `offers.json`/`roaming.json` to `data/backups/<ts>/` and writes **drafts** (`*.generated.json`); `score_catalog` then compares them to your verified gold.
- **Open-source only** (Theme-5 brief): Qwen-32B is open; Groq's free tier is used here for a one-time DEV validation, not production.

> One-time: the repo is private, so add a read-only GitHub token as a Colab secret named **`GH_TOKEN`** (key icon → Add new secret → enable Notebook access).

## Step 1 — Pull the code + use the cached scrape (no re-crawl, no GPU)

In [ ]:
# Pulls the private repo (which already contains the cached scrape and the gold catalogues).
# No pip install, no torch: the Groq path uses only the standard library + a hosted model.
import os, sys, shutil, subprocess, json
from google.colab import userdata

OWNER, REPO_NAME = "YacefMehdi", "DjezzyBot"
PROJ = "/content/DjezzyBot"
CLEAN_URL = f"https://github.com/{OWNER}/{REPO_NAME}.git"
try:
    token = userdata.get("GH_TOKEN")
except Exception as e:
    raise SystemExit("Add the GH_TOKEN Colab secret (key icon ▸ add GH_TOKEN ▸ enable "
                     f"Notebook access) and re-run. ({type(e).__name__})")
auth_url = f"https://{token}@github.com/{OWNER}/{REPO_NAME}.git"

def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError((r.stderr or r.stdout).replace(token, "***"))
    return r.stdout

if os.path.isdir(os.path.join(PROJ, ".git")):
    _run(f"git -C {PROJ} reset --hard -q")
    _run(f"git -C {PROJ} pull -q {auth_url} main")
else:
    if os.path.isdir(PROJ):
        shutil.rmtree(PROJ)
    _run(f"git clone -q {auth_url} {PROJ}")
    _run(f"git -C {PROJ} remote set-url origin {CLEAN_URL}")
os.chdir(PROJ); sys.path.insert(0, PROJ)

print("Code:", _run(f"git -C {PROJ} log -1 --oneline").strip())
print("Cached scrape:", len(json.load(open("data/djezzy_pages.json", encoding="utf-8"))),
      "pages (no re-crawl needed)")

## Step 2 — Extract the catalogue with the bigger open Qwen (drafts only)

The key stays in this notebook (typed via `getpass`, never stored). Get a free one at **console.groq.com ▸ API Keys**. Your gold is hashed before and after to prove the run leaves it untouched.

In [ ]:
import os, getpass, importlib, hashlib

os.environ["LLM_BACKEND"]  = "api"
os.environ["LLM_API_BASE"] = "https://api.groq.com/openai/v1"
os.environ["LLM_MODEL"]    = "qwen-2.5-32b"      # confirm the id if it errors (Step 2b)
os.environ["LLM_API_KEY"]  = getpass.getpass("Groq API key (free, console.groq.com): ")

h = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
gold_before = (h("data/offers.json"), h("data/roaming.json"))     # snapshot your gold

import build_catalog, score_catalog
importlib.reload(build_catalog); importlib.reload(score_catalog)  # re-read LLM_BACKEND=api
build_catalog.main()        # backup + *.generated.json drafts (gold not overwritten)
ok = score_catalog.main()   # accuracy vs your verified gold: MATCH / MISSING / PHANTOM

gold_after = (h("data/offers.json"), h("data/roaming.json"))
print("\nYour gold offers.json / roaming.json untouched by the extraction:",
      gold_before == gold_after)

### Step 2b (only if Step 2 says "model not found")
Lists the Qwen ids your key can use; set `LLM_MODEL` to one of them and re-run Step 2.

In [ ]:
import json, urllib.request, os
req = urllib.request.Request("https://api.groq.com/openai/v1/models",
        headers={"Authorization": f"Bearer {os.environ['LLM_API_KEY']}"})
print([m["id"] for m in json.load(urllib.request.urlopen(req))["data"] if "qwen" in m["id"].lower()])

## Step 3 — Why the daily scraper can never corrupt the clean catalogue

The daily refresh path is **scraper → indexer → scheduler**. None of them open or write `offers.json` / `roaming.json` — only `build_catalog.py` (this test) does. The cell below proves it from the code itself.

In [ ]:
import subprocess
hits = subprocess.run(
    "grep -nE 'offers\\.json|roaming\\.json|build_catalog' scraper.py indexer.py scheduler.py",
    shell=True, capture_output=True, text=True).stdout.strip()
print("Catalogue references in the daily-refresh code (scraper / indexer / scheduler):\n")
print(hits or "  NONE → the daily scraper + indexer rebuild only the raw pages + FAISS index;\n"
               "  they never touch your curated offers.json / roaming.json.\n\n"
               "  So the nightly refresh keeps the GENERAL answers fresh, while the clean\n"
               "  catalogue is updated ONLY by this extractor — and only on demand.")